# 📄 Descripción general del código: Evaluación de Originalidad con Embeddings
Este script implementa un sistema automático para calcular la originalidad de respuestas textuales dentro de solicitudes de apoyo a emprendimientos, usando modelos de lenguaje.

Se usa el modelo multilingüe gte-multilingual-base (basado en Transformer) para calcular similitudes semánticas entre postulaciones del mismo rubro (actividad económica) y bajo el mismo criterio.

# 🔧 Paso a paso del flujo del código:
Instalación e importación de librerías necesarias
Se instalan transformers, torch, scikit-learn, entre otros, y se importan módulos clave.

Carga del modelo
Se carga el modelo gte-multilingual-base y su tokenizer a la GPU (si está disponible).

Función de embedding por lote
Se define embed_sentences() que convierte oraciones en vectores de embedding usando el modelo cargado.

Carga de datos desde archivo Excel
Se lee el archivo que contiene columnas como combined_text, applicant_id, criteria, etc.

Generación de embeddings únicos
Se genera un embedding para cada texto único en combined_text y se asigna a la fila correspondiente.

Cálculo de vector representativo por grupo
Para cada combinación única de applicant_id + criterio, se calcula un vector promedio representativo del grupo.

Evaluación de originalidad
Para cada grupo:

Se buscan otros postulantes del mismo rubro (economic_activity) bajo el mismo criterio.

Se calcula la similitud promedio entre su vector representativo y los de sus "peers".

La originalidad se define como 1 - similitud promedio, es decir, mientras menos similar sea, más original es.

Filtrado por longitud mínima de texto
Solo se evalúan textos con más de 150 tokens para asegurar suficiente información.

Promediado por postulante y normalización
Se calcula el promedio de originalidad por postulante. Luego, se normalizan los puntajes con MinMaxScaler.

Exportación de resultados
Los puntajes finales (normalizados) se guardan en un archivo .xlsx en Google Drive.

# ✅ Resultado esperado:
Un archivo Excel con la puntuación de originalidad normalizada por cada postulante, útil para priorizar ideas más diferenciadas en procesos de evaluación automatizada.

In [ ]:
# ✅ Instalación de librerías

!pip install transformers torch scikit-learn pandas matplotlib openpyxl -q

In [ ]:
# ✅ Importación de librerías

from transformers import AutoTokenizer, AutoModel
import torch
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import drive
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler

In [ ]:
# ✅ Montar Google Drive

drive.mount('/content/drive')

In [ ]:
# ✅ Cargar modelo (uso de GPU si está disponible)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "Alibaba-NLP/gte-multilingual-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)
model.eval()

In [ ]:
# ✅ Función de embedding para oraciones (por lotes + GPU + salida en numpy)

def embed_sentences(sentences, model, tokenizer, max_length=512, batch_size=16):
    embeddings = []
    for i in tqdm(range(0, len(sentences), batch_size), desc="Embedding all sentences"):
        batch = sentences[i:i+batch_size]
        inputs = tokenizer(batch, padding=True, truncation=True, return_tensors="pt", max_length=max_length)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs)
            batch_embeds = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
            embeddings.extend(batch_embeds)
    return np.array(embeddings)

In [ ]:
# ✅ Cargar los datos

file_path = "data/grouped_categorized.xlsx"
df = pd.read_excel(file_path)
df = df.dropna(subset=['combined_text'])

In [ ]:
# ✅ Generar embeddings para todos los textos únicos

unique_texts = df['combined_text'].unique().tolist()
embeddings = embed_sentences(unique_texts, model, tokenizer)
embedding_dict = dict(zip(unique_texts, embeddings))

In [ ]:
# ✅ Añadir embeddings al DataFrame

df['embedding'] = df['combined_text'].map(embedding_dict)

In [ ]:
# ✅ Definir GroupID (Postulante + criterio)

df['GroupID'] = df['applicant_id'].astype(str) + '_' + df['criteria']

In [ ]:
# ✅ Calcular número promedio de tokens

token_length = df['combined_text'].apply(lambda x: len(tokenizer.tokenize(x)))
df['token_length'] = token_length

In [ ]:
# ✅ Calcular vector representativo por criterio (promedio por grupo)

group_df = df.groupby('GroupID').agg({
    'embedding': lambda x: np.mean(np.stack(x), axis=0),
    'token_length': 'mean',
    'applicant_id': 'first',
    'economic_activity': 'first',
    'criteria': 'first'
}).reset_index()

In [ ]:
# ✅ Calcular la originalidad por criterio

scores = []
for idx, row in tqdm(group_df.iterrows(), total=len(group_df), desc="Calculating criteria-level originality"):
    if row['token_length'] < 150:
        scores.append({
            'applicant_id': row['applicant_id'],
            'economic_activity': row['economic_activity'],
            'criteria': row['criteria'],
            'OriginalityScore': np.nan
        })
        continue

    target_vec = row['embedding'].reshape(1, -1)
    peers = group_df[(group_df['economic_activity'] == row['economic_activity']) &
                     (group_df['criteria'] == row['criteria']) &
                     (group_df['GroupID'] != row['GroupID']) &
                     (group_df['token_length'] >= 150)]

    if peers.empty:
        scores.append({
            'applicant_id': row['applicant_id'],
            'economic_activity': row['economic_activity'],
            'criteria': row['criteria'],
            'OriginalityScore': np.nan
        })
        continue

    peer_vectors = np.stack(peers['embedding'].to_numpy())
    avg_sim = cosine_similarity(target_vec, peer_vectors).mean()
    originality_score = 1 - avg_sim

    scores.append({
        'applicant_id': row['applicant_id'],
        'economic_activity': row['economic_activity'],
        'criteria': row['criteria'],
        'OriginalityScore': originality_score
    })

In [ ]:
# ✅ Calcular puntuación promedio por postulante

score_df = pd.DataFrame(scores)
final_scores = score_df.groupby('applicant_id').agg({
    'OriginalityScore': 'mean',
    'economic_activity': 'first'
}).reset_index()

In [ ]:
# ✅ Normalizar y guardar resultados (solo se usa el puntaje normalizado)

scaler = MinMaxScaler()
final_scores['OriginalityScore'] = scaler.fit_transform(final_scores[['OriginalityScore']])
final_scores['OriginalityScore'] = final_scores['OriginalityScore'].apply(lambda x: "N/A" if pd.isna(x) else round(x, 4))

final_scores = final_scores[['applicant_id', 'economic_activity', 'OriginalityScore']]

output_path = "data/originality_scores.xlsx"
final_scores.to_excel(output_path, index=False)
print(f"✅ Saved (criterion-level representative-vector comparison): {output_path}")
# Spanish Version
# print(f"✅ Guardado con éxito utilizando comparación de vectores representativos por criterio: {output_path}")